# <font color="#418FDE" size="6.5" uppercase>**Expansion Binning**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Use binarization and discretization to convert continuous features into thresholded or binned representations. 
- Generate polynomial, interaction, and spline features for nonlinear modeling. 
- Build leakage-safe preprocessing steps using transformers inside pipelines. 


## **1. Thresholds and Bins**

### **1.1. Threshold Based Binarization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_01.jpg?v=1783981120" width="250">



>* Convert values into yes-or-no indicators
>* Highlight interpretable cutoffs and sudden changes

>* Thresholds reflect meaningful real-world decision boundaries
>* Binarization loses distance from the cutoff

>* Choose thresholds carefully using training data only
>* Use bins when multiple ranges matter



In [ ]:
#@title Python Code - Threshold Based Binarization

# This example turns spending into threshold indicators.
# Binarization answers whether each value crosses a cutoff.
# The output compares original and transformed feature values.

import numpy as np
import pandas as pd
from sklearn.preprocessing import Binarizer

# These values represent annual customer spending in dollars.
spending = np.array([[120], [450], [800], [1250], [2100], [3200]])

# The threshold marks customers spending more than 1000 dollars.
threshold = 1000
binarizer = Binarizer(threshold=threshold)

# Fit is included for transformer consistency, then transform applies it.
high_spending = binarizer.fit_transform(spending).astype(int)

# Validate that the transformed feature matches the original row count.
if high_spending.shape[0] != spending.shape[0]:
    raise ValueError("Transformed data should keep the same number of rows.")

# A small table makes the yes-or-no transformation easy to inspect.
result = pd.DataFrame(
    {
        "spending_usd": spending.ravel(),
        "above_1000_usd": high_spending.ravel(),
    }
)

print("Threshold based binarization with cutoff = 1000 USD")
print(result.to_string(index=False))



### **1.2. Discretizing Continuous Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_02.jpg?v=1783981118" width="250">



>* Convert continuous values into meaningful bins
>* Reveal threshold patterns for easier modeling

>* Bins capture range-based behavior changes
>* Grouping reduces outliers but loses detail

>* Choose bin boundaries to match data and goals
>* Fit bins on training data only



In [ ]:
#@title Python Code - Discretizing Continuous Features

# This example discretizes continuous temperature measurements.
# Bins turn raw numbers into interval features.
# The output compares original and binned values.

import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

# Create a small continuous feature in degrees Celsius.
temperatures = np.array([12, 15, 18, 21, 24, 27, 30, 33, 36], dtype=float)

# Scikit-learn transformers expect a two-dimensional feature matrix.
X = temperatures.reshape(-1, 1)

# Validate the simple shape before transforming.
if X.shape != (9, 1):
    raise ValueError("Expected nine temperature measurements.")

# Learn three equal-width bins from the available values.
discretizer = KBinsDiscretizer(
    n_bins=3,
    encode="ordinal",
    strategy="uniform"
)

# Fit learns bin edges, then transform assigns each value.
bin_numbers = discretizer.fit_transform(X).astype(int).ravel()

# Build readable interval labels from the learned edges.
edges = discretizer.bin_edges_[0]
labels = []

for index in range(len(edges) - 1):
    left = round(edges[index], 1)
    right = round(edges[index + 1], 1)
    labels.append(f"{left} to {right} °C")

# Show how each continuous value becomes a bin.
result = pd.DataFrame(
    {
        "temp_C": temperatures.astype(int),
        "bin_id": bin_numbers,
        "bin_range": [labels[number] for number in bin_numbers],
    }
)

print("Learned bin edges in °C:", np.round(edges, 1).tolist())
print(result.head(5).to_string(index=False))



### **1.3. Choosing Bin Strategies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_03.jpg?v=1783981122" width="250">



>* Equal-width bins split ranges into fixed intervals
>* Uneven data can create sparse, unhelpful bins

>* Quantile bins balance observations across categories
>* Boundaries may be uneven and less intuitive

>* Use meaningful, domain-aware bin cutoffs
>* Avoid leakage, overfitting, and noisy bins



In [ ]:
#@title Python Code - Choosing Bin Strategies

# Compare bin strategies for one skewed feature.
# Equal-width and quantile bins behave differently.
# The output shows bin counts and boundaries.

import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

# Create a small skewed income feature in thousands.
income_k = np.array([28, 31, 34, 36, 39, 42, 45, 49, 55, 62, 80, 150])
income_table = pd.DataFrame({"income_k": income_k})

# Validate the feature before fitting bin transformers.
if income_table.shape != (12, 1):
    raise ValueError("Expected one feature with twelve rows.")

# Equal-width bins split the numeric range evenly.
uniform_binner = KBinsDiscretizer(
    n_bins=4,
    encode="ordinal",
    strategy="uniform"
)

# Quantile bins try to place similar counts in each bin.
quantile_binner = KBinsDiscretizer(
    n_bins=4,
    encode="ordinal",
    strategy="quantile"
)

uniform_bins = uniform_binner.fit_transform(income_table).astype(int).ravel()
quantile_bins = quantile_binner.fit_transform(income_table).astype(int).ravel()

# Count how many observations land in each bin.
uniform_counts = np.bincount(uniform_bins, minlength=4)
quantile_counts = np.bincount(quantile_bins, minlength=4)

# Show compact boundaries and counts for comparison.
uniform_edges = np.round(uniform_binner.bin_edges_[0], 1)
quantile_edges = np.round(quantile_binner.bin_edges_[0], 1)

print("Income values are in thousands of dollars.")
print("Equal-width edges:", uniform_edges.tolist())
print("Equal-width counts:", uniform_counts.tolist())
print("Quantile edges:", quantile_edges.tolist())
print("Quantile counts:", quantile_counts.tolist())
print("Equal-width keeps distance; quantile balances observations.")



## **2. Feature Expansion**

### **2.1. Polynomial Feature Expansion**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_01.jpg?v=1783981124" width="250">



>* Adds curved patterns to linear models
>* Learns bends using squared or cubed features

>* Polynomial terms reveal nonlinear feature effects
>* They help simpler models capture curves

>* Avoid overfitting with excessive polynomial complexity
>* Use scaling, validation, and regularization wisely



In [ ]:
#@title Python Code - Polynomial Feature Expansion

# This example shows polynomial feature expansion.
# Squared features help linear models bend.
# The plot compares straight and curved fits.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

# A fixed generator makes the example repeatable.
rng = np.random.default_rng(42)

# One input feature has a curved relationship with cost.
machine_age = np.linspace(0, 10, 40).reshape(-1, 1)
noise = rng.normal(0, 6, size=machine_age.shape[0])
maintenance_cost = 20 + 2 * machine_age.ravel() + 1.5 * machine_age.ravel() ** 2
maintenance_cost = maintenance_cost + noise

# This check keeps the feature and target lengths aligned.
if machine_age.shape[0] != maintenance_cost.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# A plain linear model can only draw a straight line.
linear_model = LinearRegression()
linear_model.fit(machine_age, maintenance_cost)

# PolynomialFeatures creates x and x squared before regression.
polynomial_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression(),
)
polynomial_model.fit(machine_age, maintenance_cost)

# A smooth grid makes the fitted curves easy to see.
age_grid = np.linspace(0, 10, 200).reshape(-1, 1)
linear_prediction = linear_model.predict(age_grid)
polynomial_prediction = polynomial_model.predict(age_grid)

# The transformed feature names reveal the expansion.
feature_maker = polynomial_model.named_steps["polynomialfeatures"]
expanded_names = feature_maker.get_feature_names_out(["age"])

print(f"scikit-learn version: {sklearn_version}")
print(f"Original feature count: {machine_age.shape[1]}")
print(f"Expanded feature count: {len(expanded_names)}")
print(f"Expanded feature names: {list(expanded_names)}")

# The curved fit uses expanded features inside the pipeline.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(machine_age.ravel(), maintenance_cost, label="observed data", alpha=0.75)
ax.plot(age_grid.ravel(), linear_prediction, label="linear fit")
ax.plot(age_grid.ravel(), polynomial_prediction, label="degree 2 polynomial fit")
ax.set_title("Polynomial expansion lets linear regression fit a curve")
ax.set_xlabel("Machine age in years")
ax.set_ylabel("Maintenance cost")
ax.legend()
plt.show()



### **2.2. Interaction Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_02.jpg?v=1783981127" width="250">



>* Combine features to capture dependent effects
>* Reveal nonlinear patterns linear models miss

>* Combined features reveal extra predictive patterns
>* Interactions support group-specific feature effects

>* Too many interactions can add noise
>* Choose meaningful combinations using validation



In [ ]:
#@title Python Code - Interaction Features

# This example builds interaction features for linear regression.
# Interactions let one feature change another feature's effect.
# The printed scores compare models with and without interactions.

import numpy as np
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

# Create a small dataset where the true pattern is an interaction.
rng = np.random.default_rng(42)
area = rng.uniform(50, 200, size=120)
quality = rng.uniform(1, 10, size=120)

# Price depends strongly on area multiplied by neighborhood quality.
noise = rng.normal(0, 20, size=120)
price = 30 + 0.4 * area + 8 * quality + 0.25 * area * quality + noise
features = np.column_stack((area, quality))

# Check that the feature matrix has the expected two columns.
if features.shape != (120, 2):
    raise ValueError("Expected 120 rows and 2 input features.")

# Split before preprocessing to keep the transformation leakage-safe.
X_train, X_test, y_train, y_test = train_test_split(
    features, price, test_size=0.25, random_state=42
)

# Fit a plain linear model using only the original features.
plain_model = LinearRegression()
plain_model.fit(X_train, y_train)
plain_score = r2_score(y_test, plain_model.predict(X_test))

# Add only the area-by-quality interaction inside a pipeline.
interaction_model = Pipeline(
    [("interactions", PolynomialFeatures(degree=2, interaction_only=True,
                                          include_bias=False)),
     ("linear_model", LinearRegression())]
)

# Fit the pipeline only on training data, then evaluate on test data.
interaction_model.fit(X_train, y_train)
interaction_score = r2_score(y_test, interaction_model.predict(X_test))

# Show the new feature names created by PolynomialFeatures.
feature_names = interaction_model.named_steps["interactions"].get_feature_names_out(
    ["area", "quality"]
)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Original feature count: {X_train.shape[1]}")
print(f"Expanded feature count: {len(feature_names)}")
print(f"Expanded features: {', '.join(feature_names)}")
print(f"Test R2 without interaction: {plain_score:.3f}")
print(f"Test R2 with interaction: {interaction_score:.3f}")



### **2.3. Spline Feature Basis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_03.jpg?v=1783981126" width="250">



>* Split continuous features into smooth knot regions
>* Capture flexible curves without unstable extremes

>* Model smooth nonlinear feature effects
>* Keep linear models flexible and interpretable

>* Choose knot number and placement carefully
>* Fit spline preprocessing on training data only



In [ ]:
#@title Python Code - Spline Feature Basis

# This example builds smooth spline features.
# Splines let linear models learn curves.
# The plot compares raw and spline predictions.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Create one nonlinear feature with deterministic noise.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 120).reshape(-1, 1)

# The target bends smoothly across the input range.
noise = rng.normal(0, 0.25, size=x.shape[0])
y = np.sin(x[:, 0]) + 0.15 * x[:, 0] + noise

# Split before fitting preprocessing to avoid leakage.
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)

# Fit a plain linear model on the original feature.
linear_model = LinearRegression()
linear_model.fit(x_train, y_train)

# Fit spline preprocessing and regression inside one pipeline.
spline_model = make_pipeline(
    SplineTransformer(n_knots=6, degree=3, include_bias=False),
    LinearRegression(),
)
spline_model.fit(x_train, y_train)

# Validate that the spline transformer expanded one column.
spline_features = spline_model.named_steps["splinetransformer"].transform(x_train)
print("scikit-learn version:", sklearn_version)
print("Original feature columns:", x_train.shape[1])
print("Spline feature columns:", spline_features.shape[1])

# Compare test error for straight and curved fits.
linear_rmse = mean_squared_error(y_test, linear_model.predict(x_test)) ** 0.5
spline_rmse = mean_squared_error(y_test, spline_model.predict(x_test)) ** 0.5
print("Linear test RMSE:", round(linear_rmse, 3))
print("Spline test RMSE:", round(spline_rmse, 3))

# Draw predictions over a smooth grid.
x_grid = np.linspace(0, 10, 300).reshape(-1, 1)
linear_curve = linear_model.predict(x_grid)
spline_curve = spline_model.predict(x_grid)

# Plot the data and both fitted relationships.
plt.figure(figsize=(8, 5))
plt.scatter(x_train[:, 0], y_train, s=18, alpha=0.55, label="training data")
plt.plot(x_grid[:, 0], linear_curve, label="linear fit")
plt.plot(x_grid[:, 0], spline_curve, label="spline fit")
plt.title("Spline Feature Basis for a Nonlinear Relationship")
plt.xlabel("Input feature")
plt.ylabel("Target value")
plt.legend()
plt.show()



## **3. Safe Pipeline Transformations**

### **3.1. Custom Function Transformers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_01.jpg?v=1783981134" width="250">



>* Wrap custom preprocessing inside pipeline transformers
>* Ensure consistent transformations across all datasets

>* Use stable inputs, outputs, and row alignment
>* Keep transformations clear, documented, and explainable

>* Pipelines prevent leakage and inconsistent preprocessing
>* Fit learned transformations within training folds



In [ ]:
#@title Python Code - Custom Function Transformers

# Demonstrate custom transformers inside safe pipelines.
# Fit preprocessing only on training folds.
# Compare leakage-safe and manual feature engineering.

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler

# Create a small deterministic customer dataset.
rng = np.random.default_rng(42)
n_customers = 240
income = rng.normal(60000, 15000, n_customers).clip(25000, 120000)

# Monthly debt is related to income but still noisy.
debt = income * rng.uniform(0.05, 0.55, n_customers)
debt = debt + rng.normal(0, 250, n_customers)
debt = debt.clip(100, 50000)

# The target depends partly on a domain ratio.
debt_ratio = debt / income
risk_score = debt_ratio + rng.normal(0, 0.08, n_customers)
high_risk = (risk_score > 0.32).astype(int)

# Store features in a labeled table.
customers = pd.DataFrame({"income": income, "debt": debt})
labels = high_risk

# Validate the simple assumptions used by the transformer.
if customers.shape != (n_customers, 2):
    raise ValueError("Expected two numeric feature columns.")

# Split before fitting any preprocessing step.
X_train, X_test, y_train, y_test = train_test_split(
    customers,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# This function creates one domain feature from two columns.
def add_debt_ratio(data):
    table = pd.DataFrame(data, columns=["income", "debt"])
    ratio = table["debt"] / table["income"]
    return ratio.to_frame("debt_to_income")


def debt_ratio_feature_names(transformer, input_features=None):
    return np.array(["debt_to_income"], dtype=object)


# Wrap the custom function as a pipeline transformer.
ratio_transformer = FunctionTransformer(
    add_debt_ratio,
    validate=False,
    feature_names_out=debt_ratio_feature_names,
)

# Combine the custom feature with scaling inside the pipeline.
preprocess = ColumnTransformer(
    [("ratio", ratio_transformer, ["income", "debt"])],
    remainder="drop",
)

# The model receives only pipeline-produced training features.
safe_pipeline = Pipeline(
    [("features", preprocess), ("scale", StandardScaler()),
     ("model", LogisticRegression(random_state=42, max_iter=200))]
)

# Fit once on training data, then evaluate on held-out data.
safe_pipeline.fit(X_train, y_train)
predictions = safe_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

# Show the transformed feature for three test customers.
preview = safe_pipeline.named_steps["features"].transform(X_test.head(3))
preview_values = np.round(preview.ravel(), 3).tolist()

print("scikit-learn version:", sklearn.__version__)
print("Pipeline test accuracy:", round(accuracy, 3))
print("First three test debt-to-income ratios:", preview_values)
print("Custom rule stayed inside fit/predict pipeline steps.")



### **3.2. Reversing Transformations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_02.jpg?v=1783981130" width="250">



>* Map transformed features back to real units
>* Reuse training-fitted parameters safely in pipelines

>* Store reversal parameters from training data only
>* Pipelines reuse fitted values to prevent leakage

>* Some transformations lose original detail
>* Pipelines clarify reversible and irreversible steps



In [ ]:
#@title Python Code - Reversing Transformations

# This example reverses fitted scaling transformations.
# Pipelines keep training parameters safely stored.
# Printed values compare original and recovered units.

import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Small income values keep the example easy to inspect.
income_dollars = np.array([[32000], [41000], [52000], [61000], [75000], [98000]])

# The split mimics future data unseen during fitting.
train_income, test_income = train_test_split(
    income_dollars, test_size=2, random_state=42
)

# The pipeline fits the scaler only on training rows.
preprocess = Pipeline([("scale", StandardScaler())])
preprocess.fit(train_income)

# Transforming test rows reuses the stored training mean and spread.
scaled_test = preprocess.transform(test_income)
recovered_test = preprocess.inverse_transform(scaled_test)

# Recomputing a scaler on test data would leak evaluation information.
leaky_scaler = StandardScaler()
leaky_scaled_test = leaky_scaler.fit_transform(test_income)

print("scikit-learn version:", sklearn.__version__)
print("Training mean stored in pipeline: $" + str(round(preprocess["scale"].mean_[0], 2)))
print("Original test incomes: " + str(test_income.ravel().astype(int).tolist()))
print("Scaled with training parameters: " + str(np.round(scaled_test.ravel(), 2).tolist()))
print("Recovered by inverse_transform: " + str(np.round(recovered_test.ravel(), 2).tolist()))
print("Leaky scaled values from test fit: " + str(np.round(leaky_scaled_test.ravel(), 2).tolist()))



### **3.3. Preventing Data Leakage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_03.jpg?v=1783981132" width="250">



>* Keep evaluation data out of preprocessing
>* Leaky transformations inflate performance estimates

>* Fit preprocessing only on training data
>* Reuse learned settings for honest evaluation

>* Refit preprocessing within each validation fold
>* Keep validation data unseen for honest results



In [ ]:
#@title Python Code - Preventing Data Leakage

# This example compares leaky and safe preprocessing.
# Pipelines fit transformers only on training folds.
# Cross-validation scores reveal the leakage risk.

import numpy as np
import sklearn
from sklearn.datasets import make_classification

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import KBinsDiscretizer

# Create a small classification dataset for the lesson.
features, target = make_classification(
    n_samples=600,
    n_features=4,
    n_informative=2,
    n_redundant=0,
    random_state=42,
)

# Validate the basic shape before modeling.
if features.shape != (600, 4):
    raise ValueError("Expected 600 rows and 4 columns.")

# This discretizer learns bin edges from data.
discretizer = KBinsDiscretizer(
    n_bins=5,
    encode="onehot-dense",
    strategy="quantile",
)

# Leaky approach fits bins before cross-validation begins.
leaky_features = discretizer.fit_transform(features)
leaky_model = LogisticRegression(max_iter=500, random_state=42)
leaky_scores = cross_val_score(leaky_model, leaky_features, target, cv=5)

# Safe approach refits bins inside each training fold.
safe_pipeline = Pipeline(
    steps=[
        ("bins", KBinsDiscretizer(n_bins=5, encode="onehot-dense")),
        ("model", LogisticRegression(max_iter=500, random_state=42)),
    ]
)

safe_scores = cross_val_score(safe_pipeline, features, target, cv=5)

# Print concise results that compare the two workflows.
print("scikit-learn version:", sklearn.__version__)
print("Leaky preprocessing mean accuracy:", round(leaky_scores.mean(), 3))
print("Pipeline-safe mean accuracy:", round(safe_scores.mean(), 3))
print("Key idea: fit preprocessing inside the pipeline.")



# <font color="#418FDE" size="6.5" uppercase>**Expansion Binning**</font>


In this lecture, you learned to:
- Use binarization and discretization to convert continuous features into thresholded or binned representations. 
- Generate polynomial, interaction, and spline features for nonlinear modeling. 
- Build leakage-safe preprocessing steps using transformers inside pipelines. 

In the next Module (Module 5), we will go over 'Categoricals Missingness'